# 💉 Análise da Cobertura Vacinal no Brasil (2015–2023)

**Pergunta central:** Quais estados brasileiros tiveram maior queda na cobertura vacinal na última década, e existe correlação com o ressurgimento de doenças evitáveis?

**Fonte dos dados:** SI-PNI / DATASUS  
**Autor:** Gabriel Ramos da Cunha da Cruz  
**GitHub:** [GabrielCruz079](https://github.com/GabrielCruz079)

---
## 0. Imports e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Bibliotecas carregadas com sucesso ✅')

---
## 1. Carregamento dos Dados

> Os dados foram obtidos no portal do DATASUS (SI-PNI) e exportados em formato CSV.
> Acesse: http://datasus.saude.gov.br → Informações de Saúde → Imunizações

In [ ]:
# Carrega o dataset principal
df = pd.read_csv('../data/raw/cobertura_vacinal_brasil.csv', encoding='latin-1', sep=';')

print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
df.head()

---
## 2. Limpeza e Tratamento dos Dados

In [ ]:
# ── 2.1 Visão geral da qualidade dos dados ──────────────────────
print('=== Valores nulos por coluna ===')
print(df.isnull().sum())
print(f'\nDuplicatas: {df.duplicated().sum()}')
print(f'\nTipos:\n{df.dtypes}')

In [ ]:
# ── 2.2 Padronização dos nomes das colunas ──────────────────────
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('ã', 'a')
    .str.replace('ç', 'c')
    .str.replace('é', 'e')
    .str.replace('ó', 'o')
)

print('Colunas após padronização:', list(df.columns))

In [ ]:
# ── 2.3 Tratamento de tipos e valores ──────────────────────────
# Converter cobertura para numérico (pode vir como string com vírgula)
df['cobertura'] = (
    df['cobertura']
    .astype(str)
    .str.replace(',', '.')
    .str.replace('%', '')
    .str.strip()
)
df['cobertura'] = pd.to_numeric(df['cobertura'], errors='coerce')

# Remover valores impossíveis (cobertura > 200% = erro de registro)
df = df[df['cobertura'] <= 200]

# Garantir que 'ano' é inteiro
df['ano'] = df['ano'].astype(int)

# Remover linhas sem cobertura
df.dropna(subset=['cobertura'], inplace=True)

print(f'Shape após limpeza: {df.shape}')
df.describe()

In [ ]:
# ── 2.4 Salvar dados tratados ───────────────────────────────────
df.to_csv('../data/processed/cobertura_vacinal_limpo.csv', index=False)
print('Dados processados salvos em /data/processed/ ✅')

---
## 3. Análise Exploratória (EDA)

In [ ]:
# ── 3.1 Cobertura média nacional por ano ────────────────────────
media_anual = df.groupby('ano')['cobertura'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(media_anual['ano'], media_anual['cobertura'], marker='o', linewidth=2.5,
        color='#1A56A0', markersize=7)
ax.axhline(y=95, color='red', linestyle='--', linewidth=1.5, label='Meta OMS (95%)')
ax.fill_between(media_anual['ano'], media_anual['cobertura'], alpha=0.1, color='#1A56A0')
ax.set_title('Cobertura Vacinal Média Nacional (2015–2023)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Ano')
ax.set_ylabel('Cobertura Média (%)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
plt.savefig('../outputs/graficos/01_evolucao_nacional.png', bbox_inches='tight')
plt.show()
print('\nCobertura por ano:')
print(media_anual.to_string(index=False))

In [ ]:
# ── 3.2 Ranking: estados com maior queda ───────────────────────
# Comparar cobertura média de 2015 vs 2023 por estado
cob_2015 = df[df['ano'] == 2015].groupby('estado')['cobertura'].mean().rename('cob_2015')
cob_2023 = df[df['ano'] == 2023].groupby('estado')['cobertura'].mean().rename('cob_2023')

ranking = pd.concat([cob_2015, cob_2023], axis=1).dropna()
ranking['variacao_pp'] = ranking['cob_2023'] - ranking['cob_2015']  # variação em pontos percentuais
ranking = ranking.sort_values('variacao_pp')

top5_queda = ranking.head(5)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top5_queda.index, top5_queda['variacao_pp'], color='#C0392B')
ax.bar_label(bars, fmt='%.1f pp', padding=4, fontsize=10)
ax.set_title('Top 5 Estados com Maior Queda na Cobertura Vacinal\n(2015 → 2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('Variação em Pontos Percentuais (pp)')
ax.axvline(0, color='gray', linewidth=0.8)
plt.tight_layout()
plt.savefig('../outputs/graficos/02_ranking_queda_estados.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.3 Heatmap: cobertura por estado × ano ─────────────────────
pivot = df.groupby(['estado', 'ano'])['cobertura'].mean().unstack()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    pivot,
    annot=True, fmt='.0f',
    cmap='RdYlGn',
    linewidths=0.5,
    vmin=60, vmax=100,
    ax=ax,
    cbar_kws={'label': 'Cobertura (%)'}
)
ax.set_title('Heatmap: Cobertura Vacinal por Estado e Ano', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Ano')
ax.set_ylabel('Estado')
plt.tight_layout()
plt.savefig('../outputs/graficos/03_heatmap_estado_ano.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.4 Análise por vacina ──────────────────────────────────────
media_vacina = (
    df[df['ano'] == 2023]
    .groupby('vacina')['cobertura']
    .mean()
    .sort_values()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#C0392B' if x < 95 else '#1A56A0' for x in media_vacina['cobertura']]
bars = ax.barh(media_vacina['vacina'], media_vacina['cobertura'], color=colors)
ax.axvline(x=95, color='black', linestyle='--', linewidth=1.5, label='Meta OMS (95%)')
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_title('Cobertura Vacinal Média por Vacina — Brasil 2023\n(vermelho = abaixo da meta OMS)', fontsize=13, fontweight='bold')
ax.set_xlabel('Cobertura Média (%)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/graficos/04_cobertura_por_vacina.png', bbox_inches='tight')
plt.show()

---
## 4. Correlação: Cobertura Vacinal × Casos Notificados

> Dados de casos notificados obtidos no SINAN (Sistema de Informação de Agravos de Notificação)

In [ ]:
# Carrega dados de casos notificados (sarampo como exemplo)
casos = pd.read_csv('../data/raw/casos_sarampo_brasil.csv', encoding='latin-1', sep=';')
casos.columns = casos.columns.str.strip().str.lower().str.replace(' ', '_')

# Merge com dados de cobertura por estado e ano
cob_anual_estado = df.groupby(['estado', 'ano'])['cobertura'].mean().reset_index()
merged = cob_anual_estado.merge(casos, on=['estado', 'ano'], how='inner')

print(f'Shape após merge: {merged.shape}')
merged.head()

In [ ]:
# ── 4.1 Scatter: cobertura × casos notificados ──────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(merged['cobertura'], merged['casos_notificados'],
           alpha=0.6, color='#1A56A0', edgecolors='white', s=80)

# Linha de tendência
z = np.polyfit(merged['cobertura'].dropna(), merged['casos_notificados'].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(merged['cobertura'].min(), merged['cobertura'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=1.5, label='Tendência')

corr = merged[['cobertura', 'casos_notificados']].corr().iloc[0, 1]
ax.set_title(f'Cobertura Vacinal × Casos Notificados de Sarampo\nCorrelação de Pearson: {corr:.2f}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Cobertura Vacinal Média (%)')
ax.set_ylabel('Casos Notificados')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/graficos/05_correlacao_cobertura_casos.png', bbox_inches='tight')
plt.show()

print(f'\nCorrelação de Pearson entre cobertura e casos notificados: {corr:.3f}')

---
## 5. Conclusões e Insights

> **Preencha esta seção com os resultados reais após rodar a análise.**

### 📌 Principais Achados

1. **Queda nacional:** A cobertura média no Brasil caiu de **X% em 2015** para **Y% em 2023**, ficando abaixo da meta de 95% estabelecida pela OMS a partir de Z.

2. **Estados críticos:** Os estados com maior queda foram **[A, B, C]**, com redução de mais de **X pontos percentuais** no período.

3. **Vacinas em risco:** As vacinas **[X e Y]** apresentam os menores índices de cobertura em 2023, com médias abaixo de **X%** — gerando risco real de surto.

4. **Correlação com casos:** Observou-se correlação de **X** entre queda de cobertura e aumento de casos notificados de sarampo, sugerindo que estados com menor vacinação concentram mais registros da doença.

### 💡 Recomendação

Os dados indicam que campanhas focadas nos estados **[X, Y, Z]** e nas vacinas **[A, B]** teriam maior impacto na recuperação da imunização nacional.

---
*Análise realizada com dados públicos do DATASUS/SI-PNI para fins educacionais.*